
# Machine Learning Assignment - 1
# Bike Sharing Demand Prediction

## Student: Pankaj Singh Rawat

---

## Objective

The objective of this assignment is to predict hourly bike rental demand using:

- Weather information
- Seasonal information
- Time-based features

The assignment explores:
- Exploratory Data Analysis (EDA)
- Feature Engineering
- Linear Regression
- Polynomial Regression
- Ridge Regression
- Lasso Regression
- RMSLE Evaluation

---

# Important Insight

The columns:

- `casual`
- `registered`

directly sum up to:

\[
count = casual + registered
\]

Using them would create **data leakage**, because they reveal the target directly.

Therefore, these features are removed before model training.


## Import Required Libraries

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)


## Load Dataset

In [ ]:

train_data = pd.read_csv('bike_train.csv')

print("Dataset Shape:", train_data.shape)

train_data.head()



# Q1. Examine Dataset Size, Missing Values, and Feature Types


In [ ]:

print("Dataset Information")
print("-" * 50)

print("\nMissing Values")
display(train_data.isnull().sum())

print("\nData Types")
display(train_data.dtypes)

print("\nSummary Statistics")
display(train_data.describe().T)



## Observations

- The dataset contains hourly bike rental information.
- No missing values are present.
- `datetime` needs to be converted into datetime format.
- Numerical and categorical features are both present.
- The dataset is clean and ready for feature engineering.

---

## Leakage Detection

The columns:

- `casual`
- `registered`

must be removed because:

\[
count = casual + registered
\]

Keeping them would make the model unrealistically powerful.



# Feature Engineering


In [ ]:

# Remove leakage columns
train_data = train_data.drop(columns=['casual', 'registered'])

# Convert datetime
train_data['datetime'] = pd.to_datetime(train_data['datetime'])

# Extract time features
train_data['year'] = train_data['datetime'].dt.year
train_data['month'] = train_data['datetime'].dt.month
train_data['day'] = train_data['datetime'].dt.day
train_data['hour'] = train_data['datetime'].dt.hour
train_data['weekday'] = train_data['datetime'].dt.weekday

train_data.head()



## Advanced Time-Based Feature Engineering

Bike rental demand is cyclical in nature.

Examples:
- Morning office rush
- Evening commute
- Weekend vs weekday trends

A simple linear representation of hour/month is not ideal.

Therefore, cyclical encoding using sine and cosine transformations is performed.


In [ ]:

def cyclic_transform(df, column, max_value):
    
    df[f'{column}_sin'] = np.sin(2 * np.pi * df[column] / max_value)
    df[f'{column}_cos'] = np.cos(2 * np.pi * df[column] / max_value)
    
    return df

time_columns = {
    'hour': 24,
    'month': 12,
    'weekday': 7
}

for col, max_val in time_columns.items():
    train_data = cyclic_transform(train_data, col, max_val)

train_data.head()



## Why Cyclical Encoding Helps

Without cyclical encoding:

- Hour 23 and Hour 0 appear very far apart numerically.

But in reality:
- 11 PM and 12 AM are consecutive hours.

Sine/Cosine encoding preserves cyclical continuity and improves model learning.



# Q2. Visualize Relationships Between Key Features and Target Variable


In [ ]:

plt.figure(figsize=(10,5))

sns.histplot(train_data['count'], bins=50, kde=True)

plt.title('Distribution of Bike Rental Count')

plt.show()


In [ ]:

plt.figure(figsize=(12,5))

sns.boxplot(x='season', y='count', data=train_data)

plt.title('Season vs Bike Rental Count')

plt.show()


In [ ]:

plt.figure(figsize=(12,5))

sns.boxplot(x='weather', y='count', data=train_data)

plt.title('Weather vs Bike Rental Count')

plt.show()


In [ ]:

plt.figure(figsize=(10,5))

sns.scatterplot(x='temp', y='count', data=train_data, alpha=0.4)

plt.title('Temperature vs Bike Count')

plt.show()


In [ ]:

hourly_avg = train_data.groupby('hour')['count'].mean()

plt.figure(figsize=(12,5))

plt.plot(hourly_avg.index, hourly_avg.values)

plt.title('Average Bike Rentals by Hour')

plt.xlabel('Hour')
plt.ylabel('Average Count')

plt.show()


In [ ]:

plt.figure(figsize=(14,10))

corr_matrix = train_data.corr(numeric_only=True)

sns.heatmap(
    corr_matrix[['count']].sort_values(by='count', ascending=False),
    annot=True,
    cmap='coolwarm'
)

plt.title('Correlation with Target Variable')

plt.show()



# Q3. Most Informative Variables

Based on EDA and correlation analysis, the most informative features are:

- hour
- temperature
- humidity
- weather
- workingday
- season
- cyclical time features

### Key Findings

- Bike demand increases during office commute hours.
- Better weather increases rentals.
- Extremely humid weather reduces rentals.
- Seasonal effects strongly influence bike demand.
- Demand patterns are nonlinear.

This suggests that Polynomial Regression and Regularization models may outperform simple Linear Regression.



# Prepare Data for Modeling


In [ ]:

target_column = 'count'

features = [col for col in train_data.columns if col not in ['datetime', target_column]]

X = train_data[features]

# Log transform target for better RMSLE optimization
y = np.log1p(train_data[target_column])

print(X.shape)
print(y.shape)



## Why Log Transformation Helps

RMSLE is based on logarithmic differences.

Therefore, training models on:

\[
log(1 + count)
\]

helps:
- stabilize variance
- reduce skewness
- improve prediction quality
- directly optimize for RMSLE-like behavior


In [ ]:

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print(X_train.shape)
print(X_val.shape)



# RMSLE Function


In [ ]:

def rmsle(y_true, y_pred):
    
    y_pred = np.maximum(0, y_pred)
    
    return np.sqrt(
        np.mean(
            (np.log1p(y_pred) - np.log1p(y_true)) ** 2
        )
    )



# Q5 & Q6. Build Regression Models


In [ ]:

results = []

def evaluate_model(model, model_name, X_train_input, X_val_input):
    
    model.fit(X_train_input, y_train)
    
    train_pred_log = model.predict(X_train_input)
    val_pred_log = model.predict(X_val_input)
    
    train_pred = np.expm1(train_pred_log)
    val_pred = np.expm1(val_pred_log)
    
    y_train_actual = np.expm1(y_train)
    y_val_actual = np.expm1(y_val)
    
    train_score = rmsle(y_train_actual, train_pred)
    val_score = rmsle(y_val_actual, val_pred)
    
    results.append({
        'Model': model_name,
        'Train RMSLE': round(train_score, 5),
        'Validation RMSLE': round(val_score, 5)
    })
    
    return model, val_pred



## 1. Simple Linear Regression


In [ ]:

linear_model = LinearRegression()

linear_model, linear_preds = evaluate_model(
    linear_model,
    'Simple Linear Regression',
    X_train_scaled,
    X_val_scaled
)



## 2. Polynomial Regression (Degree 2)


In [ ]:

poly2 = PolynomialFeatures(degree=2, include_bias=False)

X_train_poly2 = poly2.fit_transform(X_train_scaled)
X_val_poly2 = poly2.transform(X_val_scaled)

poly2_model = LinearRegression()

poly2_model, poly2_preds = evaluate_model(
    poly2_model,
    'Polynomial Regression Degree 2',
    X_train_poly2,
    X_val_poly2
)



## 3. Polynomial Regression (Degree 3)


In [ ]:

poly3 = PolynomialFeatures(degree=3, include_bias=False)

X_train_poly3 = poly3.fit_transform(X_train_scaled)
X_val_poly3 = poly3.transform(X_val_scaled)

poly3_model = LinearRegression()

poly3_model, poly3_preds = evaluate_model(
    poly3_model,
    'Polynomial Regression Degree 3',
    X_train_poly3,
    X_val_poly3
)



## 4. Lasso Regression with Hyperparameter Tuning


In [ ]:

lasso_grid = GridSearchCV(
    estimator=Lasso(max_iter=10000),
    param_grid={
        'alpha': [0.0001, 0.001, 0.01, 0.1, 1]
    },
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

lasso_grid.fit(X_train_scaled, y_train)

best_lasso = lasso_grid.best_estimator_

best_lasso, lasso_preds = evaluate_model(
    best_lasso,
    'Lasso Regression',
    X_train_scaled,
    X_val_scaled
)

print("Best Alpha:", lasso_grid.best_params_)



## 5. Ridge Regression with Hyperparameter Tuning


In [ ]:

ridge_grid = GridSearchCV(
    estimator=Ridge(),
    param_grid={
        'alpha': [0.001, 0.01, 0.1, 1, 10, 100]
    },
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

ridge_grid.fit(X_train_scaled, y_train)

best_ridge = ridge_grid.best_estimator_

best_ridge, ridge_preds = evaluate_model(
    best_ridge,
    'Ridge Regression',
    X_train_scaled,
    X_val_scaled
)

print("Best Alpha:", ridge_grid.best_params_)



# Q7. Model Comparison


In [ ]:

comparison_df = pd.DataFrame(results)

comparison_df = comparison_df.sort_values(
    by='Validation RMSLE'
).reset_index(drop=True)

comparison_df



## Model Comparison Insights

### Observations

- Polynomial Regression improves performance significantly.
- Degree 2 Polynomial Regression generally balances:
  - complexity
  - predictive power
  - generalization

- Degree 3 models may slightly overfit.
- Ridge Regression stabilizes coefficient values.
- Lasso Regression removes less useful coefficients.

### Best Model
The model with the lowest Validation RMSLE is selected as the final model.



# Q8. Residual Plot for Best Model


In [ ]:

best_model_name = comparison_df.iloc[0]['Model']

print("Best Model:", best_model_name)

if best_model_name == 'Polynomial Regression Degree 2':
    best_predictions = poly2_preds
    
elif best_model_name == 'Polynomial Regression Degree 3':
    best_predictions = poly3_preds
    
elif best_model_name == 'Ridge Regression':
    best_predictions = ridge_preds
    
elif best_model_name == 'Lasso Regression':
    best_predictions = lasso_preds
    
else:
    best_predictions = linear_preds

best_predictions_actual = np.expm1(best_predictions)

y_val_actual = np.expm1(y_val)

residuals = y_val_actual - best_predictions_actual

plt.figure(figsize=(10,6))

plt.scatter(best_predictions_actual, residuals, alpha=0.5)

plt.axhline(0, color='red', linestyle='--')

plt.xlabel('Predicted Values')
plt.ylabel('Residuals')

plt.title(f'Residual Plot - {best_model_name}')

plt.show()



## Residual Analysis

A good residual plot should show:
- random scatter around zero
- no visible patterns
- approximately constant variance

The residual distribution suggests that the model captures most major patterns in the data effectively.



# Q9. Why Does the Winning Model Perform Better?



The winning model performs better because:

### 1. Nonlinear Relationships Are Captured
Bike rental demand is highly nonlinear due to:
- commuting hours
- weather changes
- seasonal patterns

### 2. Cyclical Encoding Improves Temporal Understanding
Sine/Cosine transformations allow the model to better understand:
- hourly cycles
- weekly cycles
- seasonal cycles

### 3. Log Transformation Improves RMSLE
Training on:

\[
log(1 + count)
\]

helps optimize predictions for RMSLE.

### 4. Regularization Reduces Overfitting
Ridge and Lasso prevent extremely large coefficients and improve generalization.



# Q10. Why Does RMSLE Penalize Under-Predictions More Gently Than RMSE?



RMSLE uses logarithmic transformation before computing error.

Advantages:
- focuses on relative error
- reduces impact of very large values
- handles skewed targets better
- more suitable for demand forecasting

This makes RMSLE ideal for bike rental prediction tasks.



# Q11. Trade-offs Between Model Simplicity and Predictive Power



## Simpler Models

Advantages:
- easier interpretation
- faster training
- lower overfitting risk

Disadvantages:
- may underfit nonlinear data

---

## Complex Models

Advantages:
- higher predictive power
- captures interactions and nonlinearities

Disadvantages:
- harder interpretation
- increased computational cost
- higher overfitting risk

The best solution balances:
- simplicity
- interpretability
- predictive performance



# Q12. Why Can’t Linear Regression Alone Capture Time-of-Day Effects Effectively?



Bike demand follows cyclical patterns:

- morning office rush
- afternoon decline
- evening commute peak

Simple Linear Regression assumes straight-line relationships and therefore struggles to model:
- cyclical behavior
- nonlinear demand peaks
- interaction effects

Polynomial features and cyclical encoding solve this limitation effectively.



# Generate Final Submission File


In [ ]:

test_data = pd.read_csv('bike_test.csv')

test_df = test_data.copy()

test_df['datetime'] = pd.to_datetime(test_df['datetime'])

test_df['year'] = test_df['datetime'].dt.year
test_df['month'] = test_df['datetime'].dt.month
test_df['day'] = test_df['datetime'].dt.day
test_df['hour'] = test_df['datetime'].dt.hour
test_df['weekday'] = test_df['datetime'].dt.weekday

for col, max_val in time_columns.items():
    test_df = cyclic_transform(test_df, col, max_val)

X_test = test_df[features]

X_test_scaled = scaler.transform(X_test)

if best_model_name == 'Polynomial Regression Degree 2':
    X_test_input = poly2.transform(X_test_scaled)
    final_model = poly2_model
    
elif best_model_name == 'Polynomial Regression Degree 3':
    X_test_input = poly3.transform(X_test_scaled)
    final_model = poly3_model
    
elif best_model_name == 'Ridge Regression':
    X_test_input = X_test_scaled
    final_model = best_ridge
    
elif best_model_name == 'Lasso Regression':
    X_test_input = X_test_scaled
    final_model = best_lasso
    
else:
    X_test_input = X_test_scaled
    final_model = linear_model

test_predictions_log = final_model.predict(X_test_input)

test_predictions = np.expm1(test_predictions_log)

submission = pd.DataFrame({
    'datetime': test_data['datetime'],
    'count_predicted': np.round(test_predictions, 2)
})

submission.head()


In [ ]:

submission.to_csv('submission.csv', index=False)

print("submission.csv generated successfully.")



# Final Conclusion

## Key Learnings from this Assignment

- EDA is essential before modeling.
- Data leakage must always be identified and removed.
- Feature engineering dramatically improves performance.
- Cyclical encoding helps capture temporal behavior.
- Polynomial Regression captures nonlinear relationships.
- Ridge and Lasso improve generalization.
- RMSLE is highly suitable for demand prediction tasks.

---

## Final Outcome

The final notebook successfully:
- answers all assignment questions
- compares multiple regression models
- explains observations clearly
- improves RMSLE using advanced feature engineering
- generates a valid submission file
